# 第 1 週｜了解專案與資料檢查
每個檢查的順序：目的 → 你的程式 → 執行結果 → 你的判斷（用自己的話寫）。
本週只**發現並記錄**問題，不修改資料、不建模。

## 0. 環境準備

In [ ]:
import sys, json, re, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 170); pd.set_option("display.max_rows", 120)

# ---- 圖表的中文字型：先找系統內的，沒有（例如 Colab）就下載 Noto Sans TC ----
def setup_chinese_font():
    for name in ["Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei", "PingFang TC", "Heiti TC"]:
        if any(f.name == name for f in font_manager.fontManager.ttflist):
            return name
    try:
        import urllib.request
        path = Path("NotoSansTC-Regular.ttf")
        if not path.exists():
            urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf", path)
        font_manager.fontManager.addfont(str(path))
        return font_manager.FontProperties(fname=str(path)).get_name()
    except Exception:
        return None

FONT = setup_chinese_font()
plt.rcParams.update({"font.family": FONT or "DejaVu Sans", "axes.unicode_minus": False, "figure.dpi": 100,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df"})
if FONT is None:
    print("⚠ 找不到中文字型，圖上的中文可能變成方塊，不影響分析")

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. 了解專案
動手分析前，先靠官方說明把專案弄清楚。這一段還不用碰資料。

### 1.1 基本資訊
寫下：題目、資料來源（機構／網站）、資料網址或 DOI、取得日期、任務類型與理由（分類：預測類別；迴歸：預測數值；分群：沒有標準答案）、預測目標欄位（分群則寫分群欄位）、ID 欄位、預計使用的模型。

選題門檻（四項都要符合，不符合請先跟講師討論）：有官方欄位說明、單一表格、約 1,000 筆以上、有明確的預測目標（分群除外）。

### 1.2 變數說明
照官方的欄位說明，逐個變數寫下：中文說明、角色（`ID`／`特徵`／`目標`／`不使用`）、代碼說明或單位。官方沒寫清楚的，照實寫「官方沒有說明」，不要靠猜。

## 2. 載入資料
讀入資料，確認讀對了（編碼、標題列、工作表）。

## 3. 核對變數說明與資料
核對 1.2 的變數說明和實際資料：有沒有漏掉的變數、名稱打錯的、角色不對的；並列出每個變數的資料型態。

## 4. 資料大小與型態
**目的**：確認資料的規模，並找出「型態不對勁」的欄位。

**要看什麼**
- 列數、欄數和官方說明一致嗎？
- **看起來是數字、卻被讀成文字**的欄位（例如 `1,234`、含單位）。
- **疑似日期**的欄位：資料如果有時間順序，之後切分資料的方式會不同。
- 欄位名稱前後有沒有多餘空白。

### 你的判斷：大小與型態
列數與欄數和官方說明一致嗎？有哪些欄位的型態不對勁（是代碼卻讀成數字、是數字卻讀成文字、疑似日期）？有沒有需要先問講師的地方？

**我的判斷：**

## 5. 空值與隱藏的缺漏
**目的**：找出缺漏的資料。**空值為 0 不代表沒有缺漏**：有些資料用 `?`、`N/A`、`-999`、`9999` 這類「假值」代表沒有資料，不會被算進空值。是不是真的缺漏，要你判斷。

### 你的判斷：空值與隱藏缺漏
空值多不多、集中在哪些欄位？有沒有疑似隱藏的缺漏，你判斷它是真的缺漏嗎？第 2 週打算怎麼處理（刪除、補值、當成一類、保留），為什麼？

**我的判斷：**

## 6. ID 與重複的資料列
**目的**：確認每一列是不是一個獨立的紀錄。ID 有沒有重複？排除 ID 後，欄位完全相同的資料列有幾列？

欄位完全相同不等於同一個人（例如兩個帳戶都沒有消費、額度又相同），所以這週只記錄，不刪除。

### 你的判斷：ID 與重複
ID 有沒有重複？欄位完全相同的資料列，你覺得是同一個人重複登記，還是只是剛好一樣？第 2 週打算刪除還是保留，為什麼？

**我的判斷：**

## 7. 數值範圍
**目的**：找出不合理或需要特別注意的數值。

**要看什麼**
- **負值**：有些合理（例如溢繳造成的負餘額），有些不合理（例如年齡、數量）。
- **為 0 的比例**很高：0 代表「沒有」還是「缺漏」？
- **離群值**（IQR 法：超出 Q1−1.5×IQR 與 Q3+1.5×IQR）：只是提醒，不代表錯誤，先不要刪。
- 常數欄位（全部都一樣）。

代碼欄位的離群值沒有意義（例如 1／2 代表男／女，不是大小）。

### 你的判斷：數值範圍
哪些數值看起來不合理（負值、極端值、單位怪）？哪些其實合理，為什麼？為 0 的比例很高的欄位，0 代表什麼？（代碼欄位的離群值不用解讀）

**我的判斷：**

## 8. 類別欄位與代碼對照
**目的**：有些欄位雖然是數字，其實是**代碼**（例如 1＝男、2＝女）。對照官方說明，列出資料裡出現、但**官方沒有說明**的代碼和筆數；也找找文字欄位有沒有意思相同、寫法不同的值（例如「台北」「臺北」）。

看不懂的代碼不要自己猜意思，也不要隨意併進某一類，先記錄下來。

### 你的判斷：類別與代碼
哪些欄位是代碼？官方字典沒說明的代碼有哪些、各有幾筆、占多少？你打算怎麼處理（保留成獨立類別、標記為未知、先不處理…），為什麼？（記得：不自行猜測含義）

**我的判斷：**

## 9. 目標檢查
**目的**：確認「要預測的東西」適合你選的任務類型。依你的任務類型做對應的檢查：

| 任務類型 | 檢查什麼 |
|---|---|
| 分類 | 各類別的筆數與比例、有沒有太少的類別、全部猜多數類別的準確率 |
| 迴歸 | 目標的範圍、分布是否偏斜、有沒有極端值、0 值與負值 |
| 分群 | （沒有目標）你選的分群欄位：尺度差多少、是否高度相關、需不需要標準化 |

### 你的判斷：目標檢查
目標欄位（分群則是你選的分群欄位）的狀況，適合你選的任務類型嗎？（分類：有沒有類別太少、比例是否極端；迴歸：分布偏斜、極端值要不要處理；分群：尺度差多少、需不需要標準化）對之後的建模有什麼影響？

**我的判斷：**

## 10. 本週小結
用你自己的話寫。

**三個最重要的發現：**

**第 2 週要處理的問題（清單）：**

**想問講師的問題：**（沒有請寫「沒有」）